In [ ]:
from pyspark.sql.functions import col, avg, min, current_timestamp

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_geolocation_table_name = dbutils.widgets.get("raw_olist_geolocation_table")

silver_schema = dbutils.widgets.get("silver_schema")
geolocation_table_name = dbutils.widgets.get("geolocation_table")

In [ ]:
raw_olist_geolocation_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_geolocation_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{geolocation_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{geolocation_table_name} (
            geolocationZipCodePrefix STRING,
            geolocationLatitude DOUBLE,
            geolocationLongitude DOUBLE,
            geolocationCityName STRING,
            geolocationState STRING,
            processedTimestamp TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
geolocation_silver_df = (
    raw_olist_geolocation_df
    .where(col("geolocation_zip_code_prefix").isNotNull())
    .groupBy(
        col("geolocation_zip_code_prefix").cast("string").alias("geolocationZipCodePrefix")
    )
    .agg(
        avg("geolocation_latitude").cast("double").alias("geolocationLatitude"),
        avg("geolocation_longitude").cast("double").alias("geolocationLongitude"),
        min("geolocation_city").cast("string").alias("geolocationCityName"),
        min("geolocation_state").cast("string").alias("geolocationState"),
    )
    .withColumn("processedTimestamp", current_timestamp())
)

In [ ]:
geolocation_silver_df.createOrReplaceTempView("geolocation_silver_view")

spark.sql(f"""
    MERGE INTO {catalog}.{silver_schema}.{geolocation_table_name} AS target
    USING geolocation_silver_view AS source
    ON target.geolocationZipCodePrefix = source.geolocationZipCodePrefix
    WHEN MATCHED THEN
        UPDATE SET
            target.geolocationLatitude = source.geolocationLatitude,
            target.geolocationLongitude = source.geolocationLongitude,
            target.geolocationCityName = source.geolocationCityName,
            target.geolocationState = source.geolocationState,
            target.processedTimestamp = source.processedTimestamp
    WHEN NOT MATCHED THEN
        INSERT *
    """)